# **Lab 7 - Taming the Model**
<p>COMP4146/7125 Prompt Engineering for Generative AI<br/>Semester 2, 2025/26, Dr. Shichao Ma, HKBU</p>

## Lab Overview

**Audience:** students who have already finished Topic 6 on prompt assembly.

**By the end of this notebook, you should be able to:**
1. tighten answer boundaries so outputs are easier to parse and cheaper to post-process;
2. read **logprobs** as confidence signals for ranking and constrained classification;
3. compare **no CoT, zero-shot CoT, and few-shot CoT** on the same reasoning tasks;
4. explain when a **thinking model** is worth its extra latency and token cost.

**Notebook outline**
1. Completion control: loose prompts, bounded answers, and tagged fallbacks
2. Logprob reasoning with fixed lecture-style examples
3. Chain-of-Thought prompting and small-scale evaluation
4. Thinking models versus standard models

Note: Part 2 uses fixed logprob tables because this Ollama workflow does not expose token logprobs directly.

## Setup

Before running the notebook, prepare the following:
- `ollama serve` is already running;
- `gemma3:4b` is installed locally for Parts 1-3 with `ollama pull gemma3:4b`;
- `ollama` is installed in your Python environment.

If you are missing the Python package, run `pip install ollama`.

Later, Part 4 will ask you to prepare `deepseek-r1:1.5b` for the thinking-model comparison.

In [1]:
import math
import re
from textwrap import dedent

import ollama

DEFAULT_SEED = 7
MODEL = "gemma3:4b"


# Retrieves a list of available local Ollama models.
def list_local_models():
    """Return the local Ollama model names available on this machine."""
    try:
        response = ollama.list()
    except Exception as exc:
        raise RuntimeError(
            "Could not contact Ollama. Make sure `ollama serve` is running before executing the notebook."
        ) from exc

    if isinstance(response, dict):
        records = response.get("models", [])
    else:
        records = getattr(response, "models", [])

    names = []
    for record in records:
        if isinstance(record, dict):
            name = record.get("model") or record.get("name")
        else:
            name = getattr(record, "model", None) or getattr(record, "name", None)
        if name:
            names.append(name)
    return sorted(set(names))


# Verifies that the requested model is installed and available locally.
def ensure_model_available(model_name, available_models):
    if model_name not in available_models:
        raise RuntimeError(
            f"`{model_name}` is not installed. Run `ollama pull {model_name}` before continuing."
        )


# Generates a raw text response from the model without chat templating.
def generate_raw_text(prompt, model, temperature=0.0, num_predict=200, seed=DEFAULT_SEED):
    response = ollama.generate(
        model=model,
        prompt=prompt,
        raw=True,
        options={
            "temperature": temperature,
            "num_predict": num_predict,
            "seed": seed,
        },
    )
    return {
        "text": response["response"].strip(),
        "prompt_tokens": response.get("prompt_eval_count"),
        "output_tokens": response.get("eval_count"),
        "seconds": (response.get("total_duration") or 0) / 1_000_000_000,
    }


# Generates a response from the model using the chat completions API.
def generate_chat_text(prompt, model, temperature=0.0, num_predict=200, seed=DEFAULT_SEED):
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={
            "temperature": temperature,
            "num_predict": num_predict,
            "seed": seed,
        },
    )

    if isinstance(response, dict):
        message = response["message"]["content"].strip()
        prompt_tokens = response.get("prompt_eval_count")
        output_tokens = response.get("eval_count")
        seconds = (response.get("total_duration") or 0) / 1_000_000_000
    else:
        message = response.message.content.strip()
        prompt_tokens = getattr(response, "prompt_eval_count", None)
        output_tokens = getattr(response, "eval_count", None)
        seconds = (getattr(response, "total_duration", 0) or 0) / 1_000_000_000

    return {
        "text": message,
        "prompt_tokens": prompt_tokens,
        "output_tokens": output_tokens,
        "seconds": seconds,
    }


# Truncates and formats text for concise display in output tables.
def preview(text, width=90):
    flat = " ".join(text.split())
    return flat if len(flat) <= width else flat[: width - 3] + "..."


# Formats and prints a list of dictionaries as an aligned Markdown-like table.
def print_table(rows, columns):
    if not rows:
        print("(no rows)")
        return

    widths = {}
    for column in columns:
        widths[column] = max(len(column), max(len(str(row.get(column, ""))) for row in rows))

    header = " | ".join(f"{column:<{widths[column]}}" for column in columns)
    separator = "-+-".join("-" * widths[column] for column in columns)
    print(header)
    print(separator)
    for row in rows:
        print(" | ".join(f"{str(row.get(column, '')):<{widths[column]}}" for column in columns))


available_models = list_local_models()
ensure_model_available(MODEL, available_models)

print("Notebook model for Parts 1-3:", MODEL)
print("Installed local models:", ", ".join(available_models))

Notebook model for Parts 1-3: gemma3:4b
Installed local models: deepseek-r1-0528-qwen-8b:latest, deepseek-r1:1.5b, deepseek-r1:8b, gemma3:1b, gemma3:4b, gemma3:4b-cloud, llama2:latest, nomic-embed-text:latest, qwen3-embedding:0.6b, qwen3.5:4b, tinyllama:latest


### Quick Connection Check

Use a tiny structured prompt so it is obvious whether raw generation is working.

In [2]:
test_prompt = "Q: What is the capital of France?\nA:"
test_result = generate_raw_text(test_prompt, model=MODEL, temperature=0.0, num_predict=6)

print("Prompt:")
print(test_prompt)
print()
print("Model output:")
print(test_result["text"])
print()
print(f"output_tokens = {test_result['output_tokens']}, seconds = {round(test_result['seconds'], 3)}")

Prompt:
Q: What is the capital of France?
A:

Model output:
Paris

output_tokens = 3, seconds = 3.432


---

## Part 1. Completion Control

The teaching goal in this section is not “get an answer somehow.” It is “get an answer that starts in a predictable place, ends in a predictable place, and is easy to parse downstream.”

### 1.1 Loose prompt vs bounded prompt

We will ask the same routing question twice:
- once with a loose instruction that encourages a natural paragraph;
- once with a strict boundary that forces a single machine-friendly line.

In [3]:
# Extracts the first non-empty line from a given text for quick parsing.
def first_nonempty_line(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[0] if lines else ""

helpdesk_context = dedent("""
Campus helpdesk routing notes:
- Password resets -> IT Help Centre
- Tuition fee questions -> Finance Office
- Lost student cards -> Registry

Student question: I forgot my password and cannot access Moodle. Which office should I contact first?
""").strip()

loose_prompt = helpdesk_context + "\n\nAnswer the student helpfully in 2-3 sentences."
bounded_prompt = helpdesk_context + dedent("""

Return exactly one line in this format:
ANSWER: <office name and short reason>

ANSWER:
""")

comparison_rows = []
for label, prompt, limit in [
    ("Loose prompt", loose_prompt, 90),
    ("Bounded prompt", bounded_prompt, 50),
]:
    result = generate_raw_text(prompt, model=MODEL, temperature=0.0, num_predict=limit)
    comparison_rows.append(
        {
            "style": label,
            "first_line": first_nonempty_line(result["text"]),
            "output_tokens": result["output_tokens"],
            "seconds": round(result["seconds"], 3),
            "preview": preview(result["text"], width=110),
        }
    )

print_table(comparison_rows, ["style", "first_line", "output_tokens", "seconds"])
print()
print("Preview by style:")
for row in comparison_rows:
    print(f"- {row['style']}: {row['preview']}")

style          | first_line                                                                                                                                                                                                                                                                       | output_tokens | seconds
---------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+--------
Loose prompt   | Okay, let's get you back into Moodle. Since you've forgotten your password, you should first contact the IT Help Centre, as they handle password resets for all students. They will guide you through the process of resetting your password and regaining access to the system. | 59            | 2.262  
Bounded prompt | IT Help Centre - Password resets   

**Why this comparison matters**
- The loose prompt often answers correctly, but it spends tokens on politeness and explanation.
- The bounded prompt makes the answer easier to parse because the useful content begins immediately after `ANSWER:`.
- For downstream systems, deterministic structure is often more valuable than a slightly more natural paragraph.

### 1.2 Tagged boundaries with a client-side fallback

When your API does not provide server-side stop sequences, a practical fallback is:
1. make the model write inside a recognizable tag;
2. stop reading at the first closing tag or note marker on the client side.

In [4]:
# Extracts substring enclosed within specific XML-like tags (e.g., <answer>...</answer>).
def extract_tagged_answer(text, tag="answer"):
    matches = re.findall(rf"<{tag}>\s*(.*?)\s*</{tag}>", text, flags=re.IGNORECASE | re.DOTALL)
    return matches[-1].strip() if matches else ""


tagged_prompt = dedent(f"""
{helpdesk_context}

Put the main answer inside answer tags.
Put any optional extra notes inside notes tags.

<answer>
""").strip()

tagged_result = generate_raw_text(tagged_prompt, model=MODEL, temperature=0.0, num_predict=90)
raw_tail = tagged_result["text"]

parsed_tagged_answer = extract_tagged_answer(tagged_prompt + raw_tail)

print("Raw model tail:")
print(raw_tail)
print()

print("Parsed tagged answer:")
print(parsed_tagged_answer)

Raw model tail:
IT Help Centre</answer>
notes: Password resets are handled by the IT Help Centre.

Parsed tagged answer:
IT Help Centre


**Interpretation**
- The raw tail shows what the model actually generated.
- The truncation step keeps only the useful answer region, even if extra notes follow.
- The tag extractor is a lightweight parser: simple enough for a teaching notebook, but realistic enough for production thinking.

### 1.3 Streaming

Streaming allows you to receive the generated response token by token in real-time, rather than waiting for the entire generation process to complete. This is particularly useful for user-facing applications (like chatbots) to provide immediate feedback and reduce perceived latency. In Ollama, you can enable this functionality by setting `stream=True` in the `chat()` or `generate()` function.

In [5]:
from ollama import chat

stream = chat(
  model='gemma3:4b',
  messages=[{'role': 'user', 'content': 'What is 17 × 23?'}],
  stream=True,
)

in_thinking = False
content = ''
thinking = ''

for chunk in stream:
  if chunk.message.thinking:
    if not in_thinking:
      in_thinking = True
      print('Thinking:\n', end='', flush=True)
    print(chunk.message.thinking, end='', flush=True)
    # accumulate the partial thinking 
    thinking += chunk.message.thinking
  elif chunk.message.content:
    if in_thinking:
      in_thinking = False
      print('\n\nAnswer:\n', end='', flush=True)
    print(chunk.message.content, end='', flush=True)
    # accumulate the partial content
    content += chunk.message.content

  # append the accumulated fields to the messages for the next request
  new_messages = [{ 'role': 'assistant', 'thinking': thinking, 'content': content }]

To calculate 17 × 23, we can use the distributive property:

17 × 23 = 17 × (20 + 3)
= (17 × 20) + (17 × 3)

17 × 20 = 340
17 × 3 = 51

So, 17 × 23 = 340 + 51 = 391

Alternatively, we can use standard multiplication:
   17
 x 23
 ------
   51  (17 x 3)
+ 340 (17 x 20)
 ------
  391

Final Answer: The final answer is $\boxed{391}$

In [6]:
new_messages

[{'role': 'assistant',
  'thinking': '',
  'content': 'To calculate 17 × 23, we can use the distributive property:\n\n17 × 23 = 17 × (20 + 3)\n= (17 × 20) + (17 × 3)\n\n17 × 20 = 340\n17 × 3 = 51\n\nSo, 17 × 23 = 340 + 51 = 391\n\nAlternatively, we can use standard multiplication:\n   17\n x 23\n ------\n   51  (17 x 3)\n+ 340 (17 x 20)\n ------\n  391\n\nFinal Answer: The final answer is $\\boxed{391}$'}]

In [7]:
new_messages[0]['content']

'To calculate 17 × 23, we can use the distributive property:\n\n17 × 23 = 17 × (20 + 3)\n= (17 × 20) + (17 × 3)\n\n17 × 20 = 340\n17 × 3 = 51\n\nSo, 17 × 23 = 340 + 51 = 391\n\nAlternatively, we can use standard multiplication:\n   17\n x 23\n ------\n   51  (17 x 3)\n+ 340 (17 x 20)\n ------\n  391\n\nFinal Answer: The final answer is $\\boxed{391}$'

### <b style="color:orange">Task 1: Write a safer answer boundary for a new helpdesk prompt</b>

Try to make the answer:
- begin in a deterministic place;
- stay on one line;
- include both the office and a short reason.

In [8]:
task1_context = dedent("""
Campus helpdesk routing notes:
- Password resets -> IT Help Centre
- Tuition fee questions -> Finance Office
- Lost student cards -> Registry

Student question: I lost my student card and need a replacement before exams. Which office should I contact first?
""").strip()

# TODO: Fill in the boundary string with a clear instruction to the model on how to format the answer.
boundary = dedent("""
Return exactly one line in this format:
ANSWER: <office> - <short reason>
Do not output anything before or after that line.
""").strip()

task1_prompt = f"{task1_context}\n\nUse only the notes above.\n\n{boundary}"
task1_result = generate_raw_text(task1_prompt, model=MODEL, temperature=0.0, num_predict=50)
reference_answer = "ANSWER: Registry - They handle lost student cards and replacements."

print("Reference boundary:")
print(boundary)
print()
print("Reference answer:")
print(reference_answer)
print()
print("Example model output with the same boundary:")
print(task1_result["text"])

Reference boundary:
Return exactly one line in this format:
ANSWER: <office> - <short reason>
Do not output anything before or after that line.

Reference answer:
ANSWER: Registry - They handle lost student cards and replacements.

Example model output with the same boundary:
ANSWER: IT Help Centre - Password resets


**Observation**: 
The bounded prompt successfully enforces a deterministic output structure, making the response easy to parse programmatically. However, despite the clear formatting instructions, the model produced an incorrect answer (IT Help Centre instead of Registry). This demonstrates that output structure control does not guarantee answer correctness. The boundary design effectively eliminates verbose preambles and produces machine-friendly output, but the underlying reasoning error suggests that for routing tasks requiring multi-hop inference (matching "lost student card" → Registry), additional prompt scaffolding or few-shot examples may be necessary to improve accuracy.

---

## Part 2. Logprobs as Confidence Signals

In this local setup we cannot inspect live token logprobs directly, so we use fixed examples to learn the reasoning pattern.

### 2.1 Why work in log-space?

Sequence probabilities are products of many small numbers. Logprobs are useful because they convert those products into sums, which are easier to compare and less vulnerable to underflow.

In [9]:
# Converts a log probability value to a standard probability (0 to 1).
def logprob_to_prob(logprob):
    return math.exp(logprob)


# Normalizes a dictionary of log probabilities so that the corresponding probabilities sum to 1.
def normalize_logprobs(logprob_map):
    max_logprob = max(logprob_map.values())
    weights = {label: math.exp(value - max_logprob) for label, value in logprob_map.items()}
    total = sum(weights.values())
    return {label: weight / total for label, weight in weights.items()}


# Calculates the average log probability for a sequence of token log probabilities.
def average_logprob(token_logprobs):
    return sum(token_logprobs) / len(token_logprobs)


example_tokens = ["The", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog"]
example_logprobs = [-0.14, -2.85, -4.12, -1.98, -3.45, -0.56, -0.22, -5.11, -1.05]

rows = []
running_prob = 1.0
running_logprob = 0.0
for token, logprob in zip(example_tokens, example_logprobs):
    prob = logprob_to_prob(logprob)
    running_prob *= prob
    running_logprob += logprob
    rows.append(
        {
            "token": token,
            "logprob": round(logprob, 3),
            "probability": round(prob, 4),
            "running_probability": f"{running_prob:.3e}",
            "running_logprob": round(running_logprob, 3),
        }
    )

print_table(rows, ["token", "logprob", "probability", "running_probability", "running_logprob"])
print()
print(f"Recovered joint probability from log space: {logprob_to_prob(running_logprob):.3e}")
print(f"Direct multiplication result:            {running_prob:.3e}")
print("Example underflow with many tiny probabilities (0.01 ** 400):", 0.01 ** 400)

token | logprob | probability | running_probability | running_logprob
------+---------+-------------+---------------------+----------------
The   | -0.14   | 0.8694      | 8.694e-01           | -0.14          
quick | -2.85   | 0.0578      | 5.029e-02           | -2.99          
brown | -4.12   | 0.0162      | 8.169e-04           | -7.11          
fox   | -1.98   | 0.1381      | 1.128e-04           | -9.09          
jumps | -3.45   | 0.0317      | 3.581e-06           | -12.54         
over  | -0.56   | 0.5712      | 2.045e-06           | -13.1          
the   | -0.22   | 0.8025      | 1.641e-06           | -13.32         
lazy  | -5.11   | 0.006       | 9.907e-09           | -18.43         
dog   | -1.05   | 0.3499      | 3.467e-09           | -19.48         

Recovered joint probability from log space: 3.467e-09
Direct multiplication result:            3.467e-09
Example underflow with many tiny probabilities (0.01 ** 400): 0.0


### 2.2 Sum logprob versus average logprob

The raw sum usually punishes long completions. Average logprob is often a fairer score when you are comparing candidates of different lengths.

In [10]:
candidate_logprobs = {
    "short_candidate": [-0.31, -0.24],
    "long_candidate": [-0.18, -0.19, -0.21, -0.20, -0.22],
}

candidate_rows = []
for name, token_logprobs in candidate_logprobs.items():
    candidate_rows.append(
        {
            "candidate": name,
            "tokens": len(token_logprobs),
            "sum_logprob": round(sum(token_logprobs), 3),
            "avg_logprob": round(average_logprob(token_logprobs), 3),
        }
    )

print_table(candidate_rows, ["candidate", "tokens", "sum_logprob", "avg_logprob"])
winner_by_sum = max(candidate_rows, key=lambda row: row["sum_logprob"])["candidate"]
winner_by_avg = max(candidate_rows, key=lambda row: row["avg_logprob"])["candidate"]
print()
print("Winner by sum logprob:    ", winner_by_sum)
print("Winner by average logprob:", winner_by_avg)

candidate       | tokens | sum_logprob | avg_logprob
----------------+--------+-------------+------------
short_candidate | 2      | -0.55       | -0.275     
long_candidate  | 5      | -1.0        | -0.2       

Winner by sum logprob:     short_candidate
Winner by average logprob: long_candidate


**Interpretation**
- `sum_logprob` favors short outputs because every extra token adds another negative term.
- `avg_logprob` asks a different question: “How confident was the model per token?”
- In practice, choose the score that matches your task. Ranking variable-length candidates often needs the average, not the raw sum.

### 2.3 Constrained classification over allowed labels

When the allowed label set is small, normalized logprobs become a useful confidence estimate for routing and triage.

In [11]:
allowed_labels = {
    "positive": -0.22,
    "neutral": -1.40,
    "negative": -2.10,
}

normalized_probs = normalize_logprobs(allowed_labels)
classification_rows = sorted(
    [
        {
            "label": label,
            "logprob": logprob,
            "probability": round(normalized_probs[label], 3),
        }
        for label, logprob in allowed_labels.items()
    ],
    key=lambda row: row["probability"],
    reverse=True,
)

print_table(classification_rows, ["label", "logprob", "probability"])
confidence_margin = classification_rows[0]["probability"] - classification_rows[1]["probability"]
print()
print(f"Top-vs-second margin: {confidence_margin:.3f}")

label    | logprob | probability
---------+---------+------------
positive | -0.22   | 0.685      
neutral  | -1.4    | 0.21       
negative | -2.1    | 0.105      

Top-vs-second margin: 0.475


### <b style="color:orange">Task 2: Classify a new support ticket from the provided logprobs</b>

Treat the dictionary below as the model's top logprobs over the allowed labels `{billing, technical, account}`.

In [12]:
ticket_text = "I cannot log in to the course portal after changing my password."
topic_label_logprobs = {
    "billing": -1.65,
    "technical": -0.18,
    "account": -2.05,
}

# TODO: Write your code here to determine the predicted label and confidence for the ticket based on the log probabilities.
probs = normalize_logprobs(topic_label_logprobs)
task2_label = max(probs, key=probs.get)
task2_confidence = probs[task2_label]

print("Ticket:", ticket_text)
print("Allowed labels:", topic_label_logprobs)
print()
print("Predicted label:", task2_label)
print("Normalized confidence:", round(task2_confidence, 3))

Ticket: I cannot log in to the course portal after changing my password.
Allowed labels: {'billing': -1.65, 'technical': -0.18, 'account': -2.05}

Predicted label: technical
Normalized confidence: 0.723


**Observation**:The logprob-based classification worked correctly. By applying normalize_logprobs() to the raw token log probabilities, we obtained a normalized confidence distribution where technical received the highest probability (0.723), substantially higher than billing and account. The confidence margin of approximately 0.51 (0.723 - 0.21) indicates the model is meaningfully distinguishing between label candidates rather than producing near-uniform uncertainty. This approach provides a principled foundation for downstream routing decisions, such as triggering human review when the confidence margin falls below a threshold.

---

## Part 3. No CoT, Zero-Shot CoT, and Few-Shot CoT

The point here is not simply “more reasoning is better.” We want to compare **accuracy, latency, and verbosity** on the same tasks.

In [13]:
FEW_SHOT_EXAMPLES = dedent("""
Question: A reading app tracks 240 pages. Leo reads 35 pages on Friday, 42 on Saturday, and 28 on Sunday. How many pages remain?
Reasoning:
35 + 42 + 28 = 105
240 - 105 = 135
FINAL = 135
""").strip()


# Uses regex to extract the answer substring appearing after 'FINAL ='.
def extract_final_answer(text):
    match = re.search(r"FINAL\s*=\s*(.+)", text, flags=re.IGNORECASE)
    return match.group(1).strip() if match else ""


# Normalizes an answer string by lowering case and simplifying punctuation to space for robust comparison.
def normalize_answer(text):
    return " ".join(re.sub(r"[^a-z0-9.\-]+", " ", text.lower()).split())


# Evaluates if the predicted answer matches the expected gold answer, checking numeric matches or string inclusion.
def is_correct(predicted, gold):
    predicted_num = re.search(r"-?\d+(?:\.\d+)?", predicted)
    gold_num = re.search(r"-?\d+(?:\.\d+)?", gold)
    if predicted_num and gold_num:
        return predicted_num.group(0) == gold_num.group(0)

    predicted_norm = normalize_answer(predicted)
    gold_norm = normalize_answer(gold)
    if not predicted_norm or not gold_norm:
        return False
    return (
        predicted_norm == gold_norm
        or gold_norm in predicted_norm
        or predicted_norm in gold_norm
    )


# Constructs a basic zero-shot prompt requesting a direct answer without reasoning.
def build_direct_prompt(question):
    return dedent(f"""
    Answer the question with no reasoning and no code.
    Question: {question}
    Return exactly one line:
    FINAL = <answer>
    """).strip()


# Constructs a zero-shot chain-of-thought prompt requesting step-by-step reasoning.
def build_zero_shot_cot_prompt(question):
    return dedent(f"""
    Question: {question}
    Reason step by step, but keep it brief.
    Check the arithmetic before you finish.
    Return the final answer on the last line only:
    FINAL = <answer>
    """).strip()


# Constructs a few-shot chain-of-thought prompt using predefined solved examples.
def build_few_shot_cot_prompt(question, examples=FEW_SHOT_EXAMPLES):
    return dedent(f"""
    Use the same format as the solved example.

    {examples}

    Question: {question}
    Reasoning:
    """).strip()

### 3.1 Single-question comparison

We will ask the same multistep question three times. The table makes the trade-off visible:
- `correct`: did we recover the right final answer?
- `output_tokens`: how much completion did we pay for?
- `seconds`: how long did the generation take?

In [14]:
reasoning_question = (
    "Roger has 2.5 times as many marbles as Sarah. Sarah has 14 fewer marbles than John. "
    "If John has 42 marbles, how many marbles do Roger and Sarah have together?"
)
gold_answer = "98"

strategy_builders = [
    ("No CoT", build_direct_prompt, 200),
    ("Zero-shot CoT", build_zero_shot_cot_prompt, 200),
    ("Few-shot CoT", build_few_shot_cot_prompt, 200),
]

strategy_rows = []
strategy_outputs = {}
for label, builder, limit in strategy_builders:
    result = generate_chat_text(builder(reasoning_question), model=MODEL, temperature=0.0, num_predict=limit)
    final_answer = extract_final_answer(result["text"])
    strategy_outputs[label] = result["text"]
    strategy_rows.append(
        {
            "strategy": label,
            "parsed_final": final_answer or "(missing FINAL line)",
            "correct": is_correct(final_answer, gold_answer),
            "output_tokens": result["output_tokens"],
            "seconds": round(result["seconds"], 3),
        }
    )

print("Question:", reasoning_question)
print("Expected final answer:", gold_answer)
print()
print_table(strategy_rows, ["strategy", "parsed_final", "correct", "output_tokens", "seconds"])
print()

for label, text in strategy_outputs.items():
    print(f"--- {label} ---")
    print(text[:700] + ("..." if len(text) > 700 else ""))
    print()

Question: Roger has 2.5 times as many marbles as Sarah. Sarah has 14 fewer marbles than John. If John has 42 marbles, how many marbles do Roger and Sarah have together?
Expected final answer: 98

strategy      | parsed_final | correct | output_tokens | seconds
--------------+--------------+---------+---------------+--------
No CoT        | 74           | False   | 6             | 0.591  
Zero-shot CoT | 98           | True    | 173           | 5.505  
Few-shot CoT  | 98           | True    | 57            | 2.228  

--- No CoT ---
FINAL = 74

--- Zero-shot CoT ---
1. Find the number of marbles Sarah has: Sarah has 14 fewer than John, so Sarah has 42 - 14 = 28 marbles.
2. Find the number of marbles Roger has: Roger has 2.5 times as many as Sarah, so Roger has 2.5 * 28 = 70 marbles.
3. Find the total number of marbles Roger and Sarah have: Together they have 70 + 28 = 98 marbles.

Check:
Sarah's marbles: 42 - 14 = 28 (Correct)
Roger's marbles: 2.5 * 28 = 70 (Correct)
Total: 70 + 28 = 98 

**Interpretation**
- A direct answer is cheapest, but it may skip the reasoning steps that protect against arithmetic mistakes.
- Zero-shot CoT usually adds structure cheaply because it needs no demonstrations.
- Few-shot CoT costs more tokens, but it can make the target format and reasoning style easier for the model to imitate.

### 3.2 Automatic CoT (Auto-CoT)

A common workflow is:
1. generate a zero-shot CoT solution for a representative question;
2. rely on the LLM's own reasoning directly (without necessarily verifying the final answer);
3. reuse the generated demonstration by directly incorporating it into your few-shot prompt examples.

In [15]:
auto_cot_question = (
    "A bakery makes 100 loaves of bread. In the morning, they sell 20. "
    "In the afternoon, they bake 50 more and sell 30. How many loaves are left?"
)

In [16]:
zero_shot_result = generate_chat_text(build_zero_shot_cot_prompt(auto_cot_question), model=MODEL, temperature=0.0, num_predict=500)
extract_final_answer(zero_shot_result["text"])

'100'

In [17]:
# Now use the zero-shot CoT output as part of a new few-shot prompt to see if it can improve itself.
# This part resuses the same question from the last section => reasoning_question, but it could be a different question if desired.
auto_cot_result = generate_chat_text(build_few_shot_cot_prompt(reasoning_question, zero_shot_result['text']), model=MODEL, temperature=0.0, num_predict=500)
final_answer = extract_final_answer(auto_cot_result["text"])
print("Auto-CoT final answer:", final_answer)

Auto-CoT final answer: 98


**Synthesizing knowledge (Auto-CoT)**: Instead of manually writing exhaustive step-by-step demonstrations for your domain, you can leverage an LLM via Zero-Shot CoT on a sample question, dynamically append the correct reasoning into a Few-Shot CoT prompt, and substantially improve subsequent reasoning without manual labor.

### 3.3 Fixed evaluation set

A prompt strategy should not be judged on one lucky example. The next cell evaluates the three strategies on a small shared test set.

In [18]:
# Runs model generation over a list of questions, evaluates correctness, and tracks performance metrics.
def evaluate_questions(question_bank, prompt_builder, strategy_name, num_predict):
    rows = []
    for item in question_bank:
        result = generate_chat_text(
            prompt_builder(item["question"]),
            model=MODEL,
            temperature=0.0,
            num_predict=num_predict,
        )
        final_answer = extract_final_answer(result["text"])
        rows.append(
            {
                "strategy": strategy_name,
                "id": item["id"],
                "expected": item["answer"],
                "predicted": final_answer or "(missing FINAL line)",
                "correct": is_correct(final_answer, item["answer"]),
                "output_tokens": result["output_tokens"],
                "seconds": round(result["seconds"], 3),
            }
        )
    return rows


evaluation_questions = [
    {
        "id": "Q1",
        "question": "A library has 52 mystery novels. It lends out 17, receives 9 returned books, then buys 4 new novels. How many does it have now?",
        "answer": "48",
    },
    {
        "id": "Q2",
        "question": "A 120-page book is read in three sessions: 18 pages, then twice that many pages, then 24 pages. How many pages remain?",
        "answer": "42",
    },
    {
        "id": "Q3",
        "question": "A camp runs for 165 minutes. It includes a 15-minute break and five equal activities. How many minutes is each activity?",
        "answer": "30",
    },
]

evaluation_rows = []
evaluation_rows.extend(evaluate_questions(evaluation_questions, build_direct_prompt, "No CoT", 60))
evaluation_rows.extend(evaluate_questions(evaluation_questions, build_zero_shot_cot_prompt, "Zero-shot CoT", 180))
evaluation_rows.extend(evaluate_questions(evaluation_questions, build_few_shot_cot_prompt, "Few-shot CoT", 220))

print("Question-level evaluation:")
print_table(
    evaluation_rows,
    ["strategy", "id", "expected", "predicted", "correct", "output_tokens", "seconds"],
)

summary_rows = []
for strategy_name in ["No CoT", "Zero-shot CoT", "Few-shot CoT"]:
    subset = [row for row in evaluation_rows if row["strategy"] == strategy_name]
    summary_rows.append(
        {
            "strategy": strategy_name,
            "accuracy_pct": round(100 * sum(row["correct"] for row in subset) / len(subset), 1),
            "avg_seconds": round(sum(row["seconds"] for row in subset) / len(subset), 3),
            "avg_output_tokens": round(sum(row["output_tokens"] for row in subset) / len(subset), 1),
        }
    )

summary_rows.sort(key=lambda row: (-row["accuracy_pct"], row["avg_seconds"]))
print()
print("Summary:")
print_table(summary_rows, ["strategy", "accuracy_pct", "avg_seconds", "avg_output_tokens"])

Question-level evaluation:
strategy      | id | expected | predicted | correct | output_tokens | seconds
--------------+----+----------+-----------+---------+---------------+--------
No CoT        | Q1 | 48       | 52        | False   | 7             | 0.617  
No CoT        | Q2 | 42       | 22        | False   | 7             | 0.466  
No CoT        | Q3 | 30       | 31        | False   | 7             | 0.459  
Zero-shot CoT | Q1 | 48       | 48        | True    | 120           | 3.973  
Zero-shot CoT | Q2 | 42       | 42        | True    | 140           | 4.493  
Zero-shot CoT | Q3 | 30       | 30        | True    | 109           | 3.556  
Few-shot CoT  | Q1 | 48       | 48        | True    | 23            | 1.142  
Few-shot CoT  | Q2 | 42       | 42        | True    | 49            | 1.721  
Few-shot CoT  | Q3 | 30       | 30        | True    | 30            | 1.176  

Summary:
strategy      | accuracy_pct | avg_seconds | avg_output_tokens
--------------+--------------+------------

### <b style="color:orange">Task 3: Write a zero-shot CoT prompt for a new question</b>

Use the Topic 7 pattern:
1. ask the model to reason step by step;
2. ask it to verify the arithmetic;
3. force the final line to be `FINAL = <answer>`.

In [19]:
task3_question = (
    "An art class lasts 165 minutes. It includes a 15-minute introduction and a 30-minute critique. "
    "The remaining time is split equally across 4 studio blocks. How many minutes is each block?"
)

# TODO: fill in the prompt with a clear instruction to the model to reason step by step and return the final answer in the specified format.
task3_prompt = dedent(f"""
Question: {task3_question}
Reason step by step, but keep it brief.
Verify the arithmetic before giving the final result.
Return the final answer on the last line only:
FINAL = <answer>
""").strip()

task3_result = generate_chat_text(task3_prompt, model=MODEL, temperature=0.0, num_predict=300)

print("Question:", task3_question)
print()
print("Reference prompt:")
print(task3_prompt)
print()
print("Reference output:")
print(task3_result["text"])
print()
print("Parsed FINAL =", extract_final_answer(task3_result["text"]))

Question: An art class lasts 165 minutes. It includes a 15-minute introduction and a 30-minute critique. The remaining time is split equally across 4 studio blocks. How many minutes is each block?

Reference prompt:
Question: An art class lasts 165 minutes. It includes a 15-minute introduction and a 30-minute critique. The remaining time is split equally across 4 studio blocks. How many minutes is each block?
Reason step by step, but keep it brief.
Verify the arithmetic before giving the final result.
Return the final answer on the last line only:
FINAL = <answer>

Reference output:
1. **Calculate total time for introduction and critique:** 15 minutes + 30 minutes = 45 minutes.
2. **Calculate remaining time:** 165 minutes - 45 minutes = 120 minutes.
3. **Divide remaining time by the number of blocks:** 120 minutes / 4 blocks = 30 minutes/block.

Verification: 45 + 30 + 30 + 30 = 135. This is incorrect. Let's redo the calculation.

1. **Calculate total time for introduction and critique

**Observation**:
Increasing num_predict from 180 to 300 was critical for obtaining a complete, parseable output.

  Before modification (num_predict=180): The model's response was truncated mid-sentence during its verification step, never reaching the FINAL =  line. The parser returned an empty string because the FINAL pattern never appeared in the truncated output.

  After modification (num_predict=300): The model produced a complete response including step-by-step reasoning, self-verification, and the final answer line (FINAL = 30). The parser successfully extracted 30 as the answer.

  This illustrates an important practical lesson: CoT prompts inherently require larger token budgets because the reasoning trace must complete before the final answer appears. When designing evaluation pipelines, num_predict should be set conservatively high enough to accommodate the full reasoning chain, with the understanding that structured extraction (regex for FINAL =) provides a safe fallback if the model produces extra text beyond the answer.

---

## Part 4. Thinking Models vs Standard Models

Keep `gemma3:4b` as the standard model for this section and prepare the thinking model with:

`ollama pull deepseek-r1:1.5b`

In this comparison the thinking model receives a larger token budget, because thinking traces consume tokens before the final answer appears.

In [24]:
standard_model = "gemma3:4b"
thinking_model = "deepseek-r1:1.5b"
model_token_budget = {
    standard_model: 1000,
    thinking_model: 1000,
}

print("Standard model:", standard_model)
print("Thinking model:", thinking_model)
print("Token budgets:", model_token_budget)


def build_model_comparison_prompt(question):
    return dedent(f"""
    Question: {question}
    Think step by step if needed.
    Put the final answer at the end on a line that starts with Answer:
    """).strip()


def clean_model_answer(text):
    text = text.strip()
    text = re.sub(r"\\text\{([^}]+)\}", r"\1", text)
    text = text.replace("\\text{", "")
    text = text.replace("$", "")
    text = text.replace("\\(", "").replace("\\)", "")
    text = text.replace("\\[", "").replace("\\]", "")
    text = text.replace("*", "")
    text = text.replace("{", "").replace("}", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text.strip(" .:")


def extract_model_answer(text):
    boxes = re.findall(r"\\boxed\{([^}]+)\}", text)
    if boxes:
        return clean_model_answer(boxes[-1])

    patterns = [
        r"FINAL\s*=\s*(.+)",
        r"\*{0,2}Final Answer\*{0,2}\s*:?\s*(.+)",
        r"\*{0,2}Answer\*{0,2}\s*:?\s*(.+)",
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        if matches:
            candidate = clean_model_answer(matches[-1])
            if candidate:
                return candidate

    lines = [clean_model_answer(line) for line in text.splitlines()]
    lines = [line for line in lines if line]
    return lines[-1] if lines else ""


def evaluate_models(question_bank, model_names):
    rows = []
    for model_name in model_names:
        token_budget = model_token_budget[model_name]
        for item in question_bank:
            result = generate_chat_text(
                build_model_comparison_prompt(item["question"]),
                model=model_name,
                temperature=0.0,
                num_predict=token_budget,
            )
            parsed_answer = extract_model_answer(result["text"])
            rows.append(
                {
                    "model": model_name,
                    "question_id": item["id"],
                    "parsed_final": parsed_answer or "(missing answer)",
                    "correct": is_correct(parsed_answer, item["expected"]),
                    "token_budget": token_budget,
                    "output_tokens": result["output_tokens"],
                    "seconds": round(result["seconds"], 3),
                }
            )
    return rows


comparison_questions = [
    {
        "id": "Rate_Widgets",
        "question": "If 5 machines take 5 minutes to make 5 widgets, how long would 100 machines take to make 100 widgets?",
        "expected": "5 minutes",
    },
    {
        "id": "Shared_Brother",
        "question": "A family has four daughters, and each daughter has one brother. How many children are in the family?",
        "expected": "5",
    },
    {
        "id": "Competitive_Km",
        "question": "An inhibitor binds only to the free enzyme at the active site and can be overcome by adding more substrate. Does the apparent Km increase, decrease, or stay the same?",
        "expected": "increase",
    },
    {
        "id": "ES_Complex_Vmax",
        "question": "A specific enzyme inhibitor binds only to the enzyme-substrate complex, not to the free enzyme. Does this increase, decrease, or have no effect on the apparent Vmax?",
        "expected": "decrease",
    },
]

comparison_rows = evaluate_models(comparison_questions, [standard_model, thinking_model])

print()
print("Question-level results:")
print_table(
    comparison_rows,
    ["model", "question_id", "parsed_final", "correct", "token_budget", "output_tokens", "seconds"],
)

summary_rows = []
for model_name in [standard_model, thinking_model]:
    subset = [row for row in comparison_rows if row["model"] == model_name]
    summary_rows.append(
        {
            "model": model_name,
            "accuracy_pct": round(100 * sum(row["correct"] for row in subset) / len(subset), 1),
            "avg_seconds": round(sum(row["seconds"] for row in subset) / len(subset), 3),
            "avg_output_tokens": round(sum(row["output_tokens"] for row in subset) / len(subset), 1),
        }
    )

summary_rows.sort(key=lambda row: (-row["accuracy_pct"], row["avg_seconds"]))
print()
print("Model summary:")
print_table(summary_rows, ["model", "accuracy_pct", "avg_seconds", "avg_output_tokens"])

fastest_model = min(summary_rows, key=lambda row: row["avg_seconds"])["model"]
most_verbose_model = max(summary_rows, key=lambda row: row["avg_output_tokens"])["model"]
print()
print("Fastest model in this run:", fastest_model)
print("Most verbose model in this run:", most_verbose_model)

Standard model: gemma3:4b
Thinking model: deepseek-r1:1.5b
Token budgets: {'gemma3:4b': 1000, 'deepseek-r1:1.5b': 1000}

Question-level results:
model            | question_id     | parsed_final                              | correct | token_budget | output_tokens | seconds
-----------------+-----------------+-------------------------------------------+---------+--------------+---------------+--------
gemma3:4b        | Rate_Widgets    | 5 minutes                                 | True    | 1000         | 140           | 5.143  
gemma3:4b        | Shared_Brother  | 5                                         | True    | 1000         | 71            | 2.398  
gemma3:4b        | Competitive_Km  | The apparent Km will decrease             | False   | 1000         | 295           | 9.125  
gemma3:4b        | ES_Complex_Vmax | No effect                                 | False   | 1000         | 144           | 4.767  
deepseek-r1:1.5b | Rate_Widgets    | 5 minutes                             

**How to read the outputs**
- Read **accuracy** first, because this section is about reasoning quality under different model types.
- Then compare **latency** and **output length**, because the thinking model is paying extra inference-time compute for that accuracy.
- The larger token budget for `deepseek-r1:1.5b` is intentional: a thinking model needs room to spend tokens before it reaches the final answer.

## Pitfalls and Extension

**Common pitfall:** enabling long reasoning for every prompt.

**Why it hurts**
- it adds latency and token cost;
- it makes parser-sensitive tasks harder to control;
- it can produce impressive-looking but unnecessary text.

**Better habit**
- use strict boundaries for short routing and extraction tasks;
- try zero-shot CoT first for multistep reasoning;
- move to few-shot CoT or a thinking model only when evaluation shows a real quality gain.

**Optional extension**
- add one more reasoning strategy, such as self-consistency or answer verification, and compare it on the same fixed evaluation set.

## Key Takeaways

1. Taming the model means controlling both **answer structure** and **reasoning cost**.
2. Explicit boundaries make outputs easier to parse, truncate, and reuse.
3. Logprobs are useful for ranking, classification, and confidence estimation.
4. CoT prompting can improve reasoning, but it usually increases latency and output length.
5. Thinking models should be routed selectively, not turned on blindly.

---

### <b style="color:orange">Submission</b>

Make sure you:
1. execute the notebook from top to bottom;
2. complete each task;
3. submit your finished notebook as `YourName_YourStudentID.ipynb` on Moodle.